# 15 · 실시간 작업공간 + top-hat 국소대비 검출

14의 실시간 작업공간에 **top-hat(국소 대비) 검출**을 통합. 절대높이 임계의 약점(낮은 물체 놓침, 평면 드리프트 가짜블롭)을 국소 대비로 해결. 가상 3D 씬은 **실제 카메라 포즈 방향에 맞춰** 앞뒤·좌우가 일치하며, **인터랙티브 3D 뷰어**(마우스 회전/확대/이동)도 제공.

**검출 파이프라인 (`detect="tophat"`, 이제 기본값)**
1. DA 높이맵(앵커 평면 기준)
2. `top-hat = 높이 − 국소배경(롤링볼)` → 주변보다 솟은 곳만 (저융기 포착, 저주파 드리프트 제거)
3. 후보 = `top-hat > tophat_mm` ∧ 보드XY(둘레 안쪽 축소 `xy_margin_mm<0`)
4. **봉우리 검증** = `peak > peak_min_mm` ∧ `편차(peak−median) > spread_min_mm` → 평평한 경계/드리프트 아티팩트 거부
5. 측정 = 앵커 절대높이로 크기·자세(탐지≠측정 분리). 자세는 세로/가로 비율로 서있음/누움 판별.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import cv2, numpy as np, matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, workspace as ws, scene3d as s3, depth_volume as dv, live_da as ld
OUTPUT_DIR = os.path.join(ROOT, 'output')

corners_map, markers_list, REF, L = ws.load_marker_map(os.path.join(OUTPUT_DIR, 'marker_map.json'))
x0, y0, x1, y1 = ws.map_extent_m(corners_map)
PLANE = (x0, y0, x1, y1); ORIGIN = (x0*1000, y0*1000); WS_SZ = ((x1-x0)*1000+60, (y1-y0)*1000+60)
print(f'작업공간 {(x1-x0)*1000:.0f} x {(y1-y0)*1000:.0f} mm, 마커 {len(corners_map)}개')
pipe = dv.load_depth_model()

def process(path, intr):
    K, dist = au.load_intrinsics(intr)
    frame = cv2.imread(path)
    return (frame,) + ld.process_frame_workspace(frame, pipe, K, dist, corners_map,
                                                 markers_list, PLANE, min_area_px=1200)

def demo(path, intr, title):
    frame, vis, objs, ok, n, err, pose = process(path, intr)
    print(f'[{title}] localize {n}m {err:.1f}px | 물체 {len(objs)}개')
    for o in objs:
        c = o['cyl'][0]; print(f"   {o['type']:5s} ({c[0]:+.0f},{c[1]:+.0f})mm {o['label']}")
    scene = s3.render_virtual_scene(objs, markers=markers_list, ws=WS_SZ, origin_mm=ORIGIN,
                                    title=title, cam_pose=pose, plane_xyxy=PLANE)  # 실제 카메라 방향
    camh = cv2.resize(vis, (int(vis.shape[1]*scene.shape[0]/vis.shape[0]), scene.shape[0]))
    view = np.hstack([camh, scene])
    plt.figure(figsize=(16, 5)); plt.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title(f'{title}: camera | virtual 3D'); plt.show()
    return objs, pose

## 1) 캔 + 누운 박스 (웹캠 사진)

서 있는 캔과 **누운 낮은 박스**를 동시에. 절대높이 방식은 박스를 놓쳤지만 top-hat은 둘 다 잡는다. 가상 3D도 실제처럼 좌우·앞뒤 일치.

In [ ]:
_ = demo(os.path.join(OUTPUT_DIR, 'ws_raw_000.png'),
         os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'), 'can_box')

## 2) 립밤 (폰 사진 3장)

원거리(전체보드)·근접 2장. 원거리 립밤도 top-hat으로 포착(절대높이론 놓쳤던 것). 마커·경계 오검 0.

In [ ]:
for i, f in enumerate(sorted(glob.glob(os.path.join(ROOT, 'data', 'scene_images', '31_marker', '*.jpg')))):
    _ = demo(f, os.path.join(OUTPUT_DIR, 'phone_intrinsics.npz'), f'lipbalm{i}')

## 3) 인터랙티브 3D 뷰어 (마우스 회전·확대·이동)

`render_plotly`가 검출 결과를 **자체 포함 HTML**로 저장 → 브라우저에서 **드래그=회전, 휠=확대, 우클릭드래그=이동**. 물체는 solid 원통/박스, 마커 31개 + 원점(id0) 축. `cam_pose`로 초기 방향을 실제 카메라 뷰에 맞춤(plotly는 카메라 이미지와 좌우 반사 관계라 X축을 반사 보정).

> 실시간 루프에서 `[s]` 스냅 시에도 `ws_3d_###.html`이 자동 저장·열림.

In [ ]:
frame, vis, objs, ok, n, err, pose = process(os.path.join(OUTPUT_DIR, 'ws_raw_000.png'),
                                             os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
html = os.path.join(OUTPUT_DIR, 'ws_3d_interactive.html')
fig = s3.render_plotly(objs, markers=markers_list, ws=WS_SZ, origin_mm=ORIGIN, cam_pose=pose,
                       title='workspace 3D  (drag=rotate, wheel=zoom, right-drag=pan)',
                       html_path=html, open_browser=True)   # 브라우저로 열림
print('인터랙티브 3D 저장:', html, '| 물체', len(objs))
fig.show()   # 노트북 안에서도 인터랙티브로 표시

## 4) 웹캠 실시간 루프

`run_live_workspace`가 top-hat 검출을 기본으로 사용(내부 `process_frame_workspace` → `process_frame_da(detect="tophat")`, 둘레 `xy_margin_mm=-35` 기본). 가상 씬은 카메라 포즈 방향 자동. `[s]`스냅(카메라·가상 PNG + 인터랙티브 HTML 저장) `[q]`종료.

```python
Kw, distw = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
ld.run_live_workspace(Kw, distw, corners_map, markers_list, PLANE, pipe=pipe,
                      cam_index=0, calib_wh=(1920, 1080), snapshot_dir=OUTPUT_DIR, min_area_px=1200)
```

검출 파라미터(필요시 조절): `tophat_mm=4`(국소대비 임계), `peak_min_mm=10`·`spread_min_mm=6`(봉우리 검증), `xy_margin_mm=-35`(둘레 축소), `bg_ksize=121`(국소배경 커널; 최대 물체보다 커야).

In [ ]:
# 웹캠 있으면 주석 해제:
# Kw, distw = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
# ld.run_live_workspace(Kw, distw, corners_map, markers_list, PLANE, pipe=pipe,
#                       cam_index=0, calib_wh=(1920, 1080), snapshot_dir=OUTPUT_DIR, min_area_px=1200)
print('웹캠 연결 후 주석 해제')

## 정리

- **top-hat 국소대비 검출을 실시간 파이프라인에 정식 통합**(`process_frame_da(detect="tophat")` 기본, `run_live_workspace`가 사용).
- 3중 방어: **경계 축소**(보드 둘레 들뜸) + **편차 검증**(평평한 램프/단차) + **top-hat**(저융기·원거리 물체 포착).
- 자세 분류를 절대높이 → 세로/가로 비율로 개선(밑동 그림자 제외 윗부분 폭 사용) → 누운 박스도 올바로 lie.
- **가상 3D 씬을 실제 카메라 포즈 방향에 정렬**(`_lookat_aligned`) → 앞뒤·좌우가 실제 뷰와 일치.
- **인터랙티브 3D 뷰어**(`render_plotly`, 마우스 회전/확대/이동, 자체 HTML) — `cam_pose`로 초기 방향 정렬(X축 반사 보정), 스냅 시 자동 저장.
- 검증: 캔+박스 2개, 립밤 3장 각 1개, 오검 0.
- 한계: 아주 얇고 균일한 물체(종이)는 편차 작아 놓칠 수 있음(단안 DA 본질), 캔 윤곽에 그림자 스커트 잔존(측정은 mid-slice라 정확).
- 다음(선택): 하이브리드 async 부드러움, 물체 ID 추적 add/remove, 다중시점 지름, 로봇팔 핸드-아이.